In [ ]:
import pandas as pd

from pyfaidx import Fasta

from Bio import SeqIO

import matplotlib.pyplot as plt

import numpy as np

import seaborn as sns

from Bio.Seq import Seq


In [ ]:
m_location_df = pd.read_csv("circExor/resources/RNAlocate/mRNA_location_resources.csv")

m_location_df


In [ ]:
                                                     

fasta_file = "circExor/resources/ensenble/Homo_sapiens.GRCh38.cdna.all.fa"

fasta_dict = {}

with open(fasta_file, "r") as f:

    current_symbol = None

    current_seq = []

    

    for line in f:

        line = line.strip()

        if line.startswith(">"):

                     

            if current_symbol and current_symbol not in fasta_dict:

                fasta_dict[current_symbol] = "".join(current_seq)

            

                                                

            current_symbol = None

            parts = line.split(" ")

            for part in parts:

                if part.startswith("gene_symbol:"):

                    current_symbol = part.split("gene_symbol:")[1]

                    break

            

            current_seq = []

        else:

            if current_symbol is not None:

                current_seq.append(line)

            

              

    if current_symbol and current_symbol not in fasta_dict:

        fasta_dict[current_symbol] = "".join(current_seq)

                                      

                                                         

m_location_df['Sequence'] = m_location_df['RNA Symbol'].map(fasta_dict)

m_location_df


In [ ]:
                             

            

m_location_df = m_location_df.dropna(subset=['Sequence'])

                                        

m_location_df = m_location_df[~m_location_df['Sequence'].str.contains('N|n')].reset_index(drop=True)

m_location_df


In [ ]:
import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

      

m_location_df['Sequence_Length'] = m_location_df['Sequence'].str.len()

original_m_count = len(m_location_df)

                                       

m_location_df = m_location_df[m_location_df['Sequence_Length'] <= 11238].copy()

         

deleted_m = original_m_count - len(m_location_df)

print(f"m_location_df cleaning completed: deleted {deleted_m} overlong sequences; remaining {len(m_location_df)} records.\n")

        

m_location_df['Sequence_Length'] = m_location_df['Sequence'].str.len()

            

max_length = m_location_df['Sequence_Length'].max()

min_length = m_location_df['Sequence_Length'].min()

average_length = m_location_df['Sequence_Length'].mean()

print(f"--- m_location_df length statistics ---")

print(f"Minimum sequence length: {min_length}")

print(f"Maximum sequence length: {max_length}")

print(f"Average sequence length: {average_length:.2f}")

                  

bins = np.histogram_bin_edges(m_location_df['Sequence_Length'], bins=30)

bin_counts, bin_edges = np.histogram(m_location_df['Sequence_Length'], bins=bins)

          

plt.figure(figsize=(10, 6))

sns.histplot(m_location_df['Sequence_Length'], bins=bins, kde=True, color='skyblue', label='Length Distribution')

                        

plt.axvline(min_length, color='green', linestyle='--', linewidth=1.5, label=f'Min: {min_length}')

plt.axvline(max_length, color='orange', linestyle='--', linewidth=1.5, label=f'Max: {max_length}')

plt.axvline(average_length, color='red', linestyle='--', linewidth=2, label=f'Mean: {average_length:.2f}')

           

plt.title('Sequence Length Distribution of m_location_df', fontsize=16)

plt.xlabel('Sequence Length (bp)', fontsize=12)

plt.ylabel('Frequency (Count)', fontsize=12)

plt.legend()

plt.grid(axis='y', alpha=0.3)

plt.tight_layout()

         

plt.show()


In [ ]:
m_location_df['Localization'].value_counts()


In [ ]:
                              

m_location_df['tag'] = m_location_df['Localization'].isin(['Extracellular exosome', 'Extracellular vesicle']).astype(int)

               

m_location_df['tag'].value_counts()


In [ ]:
print("Label distribution before balancing:")

print(m_location_df['tag'].value_counts())

            

min_count = m_location_df['tag'].value_counts().min()

                                                     

m_location_df = m_location_df.groupby('tag').sample(n=min_count, random_state=42).reset_index(drop=True)

            

print("\nLabel distribution after balancing:")

print(m_location_df['tag'].value_counts())


In [ ]:
                             

output_fasta = "./linearDataSet/mRNA_balanced_dataset.fa"

with open(output_fasta, "w") as f:

    for index, row in m_location_df.iterrows():

                                  

        header = f">{row['RNA Symbol']}_{row['tag']}"

        sequence = row['Sequence']

        f.write(f"{header}\n{sequence}\n")
